# Review And Clean Flagged Anomalies

This notebook helps you:

- load `models/anomaly_candidates.csv` and `models/eval_results.csv`
- compare each flagged sample with a matching good sample from the same label
- review signal plots and summary stats before deciding
- delete or quarantine flagged samples (with safe dry-run first)

Recommended flow:

1. Run all setup cells.
2. Use `review_candidate(idx)` to inspect one anomaly at a time.
3. Use `delete_candidate(idx, permanent=False)` to quarantine, or `permanent=True` to hard-delete.
4. Re-run evaluation after cleanup.


In [8]:
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

# Force inline rendering in Jupyter/Cursor notebooks.
_ip = globals().get("get_ipython", lambda: None)()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")

plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 120


def detect_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / ".git").exists() and (candidate / "scripts").exists():
            return candidate
    if (cur.parent / "scripts").exists():
        return cur.parent
    return cur


REPO_ROOT = detect_repo_root(Path.cwd())
ANOMALY_PATH = REPO_ROOT / "models" / "anomaly_candidates.csv"
EVAL_PATH = REPO_ROOT / "models" / "eval_results.csv"
QUARANTINE_ROOT = REPO_ROOT / "data" / "quarantine"
FEATURES = ["ax", "ay", "az", "gx", "gy", "gz"]

print("Notebook cwd     :", Path.cwd())
print("Detected repo    :", REPO_ROOT)
print("Anomaly path     :", ANOMALY_PATH)
print("Eval path        :", EVAL_PATH)
print("Anomaly exists   :", ANOMALY_PATH.exists())
print("Eval exists      :", EVAL_PATH.exists())


Notebook cwd     : /home/rahul/inertialink/notebooks
Detected repo    : /home/rahul/inertialink
Anomaly path     : /home/rahul/inertialink/models/anomaly_candidates.csv
Eval path        : /home/rahul/inertialink/models/eval_results.csv
Anomaly exists   : True
Eval exists      : True


In [9]:
def ensure_anomaly_file(anomaly_path: Path, eval_path: Path):
    if anomaly_path.exists():
        return
    if not eval_path.exists():
        raise FileNotFoundError(
            f"Missing {eval_path}. Run: python3 scripts/eval_model.py --source both"
        )

    cmd = ["python3", "scripts/flag_anomalies.py", "--source", "both", "--only-wrong", "1"]
    print("[info] anomaly file missing. Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def to_abs_data_path(p: str | Path) -> Path:
    p = Path(p)
    if p.is_absolute():
        return p
    return (REPO_ROOT / p).resolve()


def load_tables(anomaly_path=ANOMALY_PATH, eval_path=EVAL_PATH):
    ensure_anomaly_file(anomaly_path, eval_path)

    anom = pd.read_csv(anomaly_path)
    eval_df = pd.read_csv(eval_path, comment="#")

    required_eval = {"word", "predicted", "correct", "confidence", "source", "path", "n_frames"}
    missing = sorted(required_eval - set(eval_df.columns))
    if missing:
        raise ValueError(f"eval_results.csv missing columns: {missing}")

    # Normalize key columns to stable string forms so joins/filters work.
    for col in ["word", "predicted", "source", "sample"]:
        if col in eval_df.columns:
            eval_df[col] = eval_df[col].astype(str).str.strip()
        if col in anom.columns:
            anom[col] = anom[col].astype(str).str.strip()

    eval_df["correct_bool"] = eval_df["correct"].astype(str).str.lower().isin(["true", "1"])
    eval_df["confidence"] = pd.to_numeric(eval_df["confidence"], errors="coerce").fillna(0.0)
    eval_df["n_frames"] = pd.to_numeric(eval_df["n_frames"], errors="coerce").fillna(0)
    eval_df["path_abs"] = eval_df["path"].apply(to_abs_data_path)

    if "path" in anom.columns:
        anom["path_abs"] = anom["path"].apply(to_abs_data_path)

    return anom, eval_df


anomaly_df, eval_df = load_tables()
print(f"Flagged anomalies: {len(anomaly_df)}")
print(f"Eval rows: {len(eval_df)}")
print("Correct rows in eval:", int(eval_df["correct_bool"].sum()))
anomaly_df.head(10)


Flagged anomalies: 5
Eval rows: 10015


,word,predicted,correct,confidence,source,sample,n_frames,z_len,spike_score,z_spike,anomaly_score,reasons,path,path_abs
0,1,A,False,95.1,seed,76,155,1.10,7.771,3.07,3,high_conf_wrong|spike_outlier,data/seed/1/sample_076.csv,/home/rahul/inertialink/data/seed/1/sample_076...
1,1,2,False,96.2,seed,59,148,0.99,11.072,5.27,3,high_conf_wrong|spike_outlier,data/seed/1/sample_059.csv,/home/rahul/inertialink/data/seed/1/sample_059...
2,1,2,False,96.5,seed,435,135,0.77,4.314,0.78,2,high_conf_wrong,data/seed/1/sample_435.csv,/home/rahul/inertialink/data/seed/1/sample_435...
3,2,3,False,97.0,seed,232,197,0.67,10.458,0.67,2,high_conf_wrong,data/seed/2/sample_232.csv,/home/rahul/inertialink/data/seed/2/sample_232...
4,1,3,False,99.1,augmented,411,97,0.15,5.621,1.64,2,high_conf_wrong,data/1/sample_411.csv,/home/rahul/inertialink/data/1/sample_411.csv


In [10]:
def read_signal(csv_path: str | Path) -> pd.DataFrame:
    p = to_abs_data_path(csv_path)
    if not p.exists():
        raise FileNotFoundError(f"Missing sample file: {p}")
    df = pd.read_csv(p)
    missing = [c for c in FEATURES if c not in df.columns]
    if missing:
        raise ValueError(f"Missing signal columns {missing} in {p}")
    return df


def choose_good_sample(anomaly_row: pd.Series, eval_table: pd.DataFrame) -> pd.Series:
    label = str(anomaly_row["word"]).strip()
    source = str(anomaly_row.get("source", "")).strip()
    n_frames = float(anomaly_row.get("n_frames", 0))

    pool = eval_table[
        (eval_table["word"].astype(str).str.strip() == label)
        & (eval_table["correct_bool"])
        & (eval_table["source"].astype(str).str.strip() == source)
    ].copy()

    if pool.empty:
        pool = eval_table[
            (eval_table["word"].astype(str).str.strip() == label)
            & (eval_table["correct_bool"])
        ].copy()
    if pool.empty:
        raise ValueError(f"No good sample found for label={label}")

    pool["len_diff"] = (pool["n_frames"].astype(float) - n_frames).abs()
    pool = pool.sort_values(["len_diff", "confidence"], ascending=[True, False])
    return pool.iloc[0]


def summarize_signal(df: pd.DataFrame) -> dict:
    x = df[FEATURES].values.astype(np.float32)
    dx = np.diff(x, axis=0)
    mag = np.linalg.norm(dx, axis=1) if len(dx) else np.array([0.0], dtype=np.float32)
    return {
        "frames": len(df),
        "mean_abs": float(np.mean(np.abs(x))),
        "std": float(np.std(x)),
        "p95_step_mag": float(np.percentile(mag, 95)),
        "max_step_mag": float(np.max(mag)),
    }


def plot_comparison(
    df_bad: pd.DataFrame,
    df_good: pd.DataFrame,
    title_bad: str,
    title_good: str,
    save_name: str | None = None,
):
    fig, axes = plt.subplots(3, 2, figsize=(15, 10), sharex=False)
    axes = axes.flatten()

    for i, col in enumerate(FEATURES):
        ax = axes[i]
        ax.plot(df_bad[col].values, label="flagged", linewidth=1.3)
        ax.plot(df_good[col].values, label="good_ref", linewidth=1.1, alpha=0.9)
        ax.set_title(col)
        ax.grid(alpha=0.25)
        if i == 0:
            ax.legend()

    fig.suptitle(f"Flagged vs Good Sample\n{title_bad}\n{title_good}", y=1.02)
    plt.tight_layout()

    # Save + display image file, similar to scripts/plot_compare.py behavior.
    out_dir = REPO_ROOT / "models" / "anomaly_plots"
    out_dir.mkdir(parents=True, exist_ok=True)
    if save_name is None:
        save_name = "comparison.png"
    out_path = out_dir / save_name

    fig.savefig(out_path, dpi=150)
    plt.close(fig)

    print(f"Saved plot: {out_path}")
    display(Image(filename=str(out_path)))
    return out_path


In [12]:
def review_candidate(idx: int, anomaly_table: pd.DataFrame = anomaly_df, eval_table: pd.DataFrame = eval_df):
    if idx < 0 or idx >= len(anomaly_table):
        raise IndexError(f"idx out of range: 0..{len(anomaly_table)-1}")

    row = anomaly_table.iloc[idx]
    good = choose_good_sample(row, eval_table)

    bad_path = row["path_abs"] if "path_abs" in row else row["path"]
    good_path = good["path_abs"] if "path_abs" in good else good["path"]
    df_bad = read_signal(bad_path)
    df_good = read_signal(good_path)

    print("=== FLAGGED SAMPLE ===")
    print(row[["word", "predicted", "confidence", "source", "sample", "anomaly_score", "reasons", "path"]].to_string())
    print("\n=== MATCHED GOOD SAMPLE ===")
    print(good[["word", "predicted", "confidence", "source", "sample", "path"]].to_string())

    print("\n=== SIGNAL SUMMARY ===")
    print("Flagged:", summarize_signal(df_bad))
    print("Good   :", summarize_signal(df_good))
    print(f"\nFrames (flagged, good): {len(df_bad)}, {len(df_good)}")

    save_name = f"compare_idx{idx}_{row['word']}_{row['sample']}.png"
    return plot_comparison(
        df_bad,
        df_good,
        title_bad=f"flagged idx={idx} sample={row['sample']} label={row['word']}",
        title_good=f"good sample={good['sample']} label={good['word']}",
        save_name=save_name,
    )

# Example:
img_path = review_candidate(0)


ValueError: No good sample found for label=1

In [6]:
def quarantine_path(original_path: str | Path) -> Path:
    src = to_abs_data_path(original_path)
    data_root = (REPO_ROOT / "data").resolve()
    try:
        rel = src.resolve().relative_to(data_root)
        return QUARANTINE_ROOT / rel
    except Exception:
        return QUARANTINE_ROOT / src.name


def delete_candidate(idx: int, permanent: bool = False, dry_run: bool = True):
    row = anomaly_df.iloc[idx]
    src = row["path_abs"] if "path_abs" in row else to_abs_data_path(row["path"])
    src = Path(src)

    if not src.exists():
        print(f"[skip] missing file: {src}")
        return

    if permanent:
        if dry_run:
            print(f"[dry-run] delete file: {src}")
            return
        src.unlink()
        print(f"[deleted] {src}")
        return

    dst = quarantine_path(src)
    if dry_run:
        print(f"[dry-run] move {src} -> {dst}")
        return

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dst))
    print(f"[quarantined] {src} -> {dst}")


def delete_flagged(indices, permanent: bool = False, dry_run: bool = True):
    for idx in indices:
        delete_candidate(int(idx), permanent=permanent, dry_run=dry_run)

# Examples:
# delete_candidate(0, permanent=False, dry_run=True)
# delete_flagged([0, 2, 4], permanent=False, dry_run=False)


In [7]:
review_table = anomaly_df.copy()
if "path_abs" in review_table.columns:
    review_table["exists"] = review_table["path_abs"].apply(lambda p: Path(str(p)).exists())
else:
    review_table["exists"] = review_table["path"].apply(lambda p: to_abs_data_path(p).exists())

review_table = review_table.sort_values(["anomaly_score", "confidence"], ascending=[False, False])
review_table.reset_index(drop=True, inplace=True)

print(f"Candidates available on disk: {review_table['exists'].sum()}/{len(review_table)}")
review_table.head(20)


Candidates available on disk: 5/5


,word,predicted,correct,confidence,source,sample,n_frames,z_len,spike_score,z_spike,anomaly_score,reasons,path,path_abs,exists
0,1,2,False,96.2,seed,59,148,0.99,11.072,5.27,3,high_conf_wrong|spike_outlier,data/seed/1/sample_059.csv,/home/rahul/inertialink/data/seed/1/sample_059...,True
1,1,A,False,95.1,seed,76,155,1.10,7.771,3.07,3,high_conf_wrong|spike_outlier,data/seed/1/sample_076.csv,/home/rahul/inertialink/data/seed/1/sample_076...,True
2,1,3,False,99.1,augmented,411,97,0.15,5.621,1.64,2,high_conf_wrong,data/1/sample_411.csv,/home/rahul/inertialink/data/1/sample_411.csv,True
3,2,3,False,97.0,seed,232,197,0.67,10.458,0.67,2,high_conf_wrong,data/seed/2/sample_232.csv,/home/rahul/inertialink/data/seed/2/sample_232...,True
4,1,2,False,96.5,seed,435,135,0.77,4.314,0.78,2,high_conf_wrong,data/seed/1/sample_435.csv,/home/rahul/inertialink/data/seed/1/sample_435...,True


In [ ]:
def review_all_flagged(
    anomaly_table: pd.DataFrame = anomaly_df,
    eval_table: pd.DataFrame = eval_df,
    display_images: bool = True,
):
    """Review every flagged sample and generate one comparison image per row."""
    results = []

    for idx in range(len(anomaly_table)):
        try:
            row = anomaly_table.iloc[idx]
            good = choose_good_sample(row, eval_table)

            bad_path = row["path_abs"] if "path_abs" in row else row["path"]
            good_path = good["path_abs"] if "path_abs" in good else good["path"]
            df_bad = read_signal(bad_path)
            df_good = read_signal(good_path)

            save_name = f"compare_idx{idx}_{row['word']}_{row['sample']}.png"
            out_path = plot_comparison(
                df_bad,
                df_good,
                title_bad=f"flagged idx={idx} sample={row['sample']} label={row['word']}",
                title_good=f"good sample={good['sample']} label={good['word']}",
                save_name=save_name,
            )

            if not display_images:
                # plot_comparison already displays image by default; this option keeps summary only.
                pass

            results.append({
                "idx": idx,
                "word": str(row.get("word", "")),
                "predicted": str(row.get("predicted", "")),
                "confidence": float(row.get("confidence", 0)),
                "source": str(row.get("source", "")),
                "sample": str(row.get("sample", "")),
                "status": "ok",
                "image_path": str(out_path),
                "flagged_path": str(row.get("path", "")),
                "good_path": str(good.get("path", "")),
            })
        except Exception as e:
            results.append({
                "idx": idx,
                "word": str(anomaly_table.iloc[idx].get("word", "")),
                "predicted": str(anomaly_table.iloc[idx].get("predicted", "")),
                "confidence": float(anomaly_table.iloc[idx].get("confidence", 0)),
                "source": str(anomaly_table.iloc[idx].get("source", "")),
                "sample": str(anomaly_table.iloc[idx].get("sample", "")),
                "status": f"error: {e}",
                "image_path": "",
                "flagged_path": str(anomaly_table.iloc[idx].get("path", "")),
                "good_path": "",
            })

    summary = pd.DataFrame(results).sort_values(["status", "idx"]).reset_index(drop=True)
    print(f"Reviewed {len(summary)} flagged samples.")
    print("OK:", int((summary["status"] == "ok").sum()), "| Errors:", int((summary["status"] != "ok").sum()))
    return summary

# Run all at once:
# all_summary = review_all_flagged()
# all_summary